<a href="https://colab.research.google.com/github/hpatel1933/AAI2025/blob/main/Exercise_1_Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for Customer Support

This notebook uses the live OpenRouter API through the Colab Secret OPENROUTER_API_KEY. It demonstrates a four-step prompt chain: classify the issue, ask for missing information, propose a solution using earlier outputs, and decide whether to escalate. The notebook shows the original and improved prompts, connected outputs, an iteration note, a smoke test, and successful visible output.

In [16]:
!pip -q install -U openai
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
OPENROUTER_MODEL = "openai/gpt-4o-mini"
customer_message = "My internet has not worked since yesterday, and I already restarted my router."
def ask_openrouter(prompt):
    return client.chat.completions.create(model=OPENROUTER_MODEL, messages=[{"role":"user","content":prompt}], temperature=0).choices[0].message.content.strip()

def run_chain(message):
    original_prompt = "Classify the support issue by category, urgency, and emotional tone. Do not invent account details, refunds, or policies. Return concise labeled fields.\nCustomer: " + message
    improved_prompt = "Classify the support issue by category, urgency, and emotional tone. Use only the message; do not invent account details, refunds, policies, outage status, or troubleshooting results. Return exactly CATEGORY, URGENCY, and TONE with one short reason each.\nCustomer: " + message
    classification = ask_openrouter(improved_prompt)
    missing_prompt = "You are step 2 of a customer-support prompt chain. Using the customer message and Step 1 classification below, ask no more than three useful follow-up questions. Do not invent account details, refunds, or policies.\nCustomer: " + message + "\nStep 1 classification: " + classification
    missing = ask_openrouter(missing_prompt)
    solution_prompt = "You are step 3. Use the customer message, classification, and missing-information analysis to provide polite, numbered troubleshooting steps. Do not claim an outage, refund, account access, or policy that is not provided.\nCustomer: " + message + "\nClassification: " + classification + "\nMissing information: " + missing
    solution = ask_openrouter(solution_prompt)
    escalation_prompt = "You are step 4. Decide ESCALATE or DO NOT ESCALATE and give a brief reason. The customer explicitly says they already restarted the router, so treat that recommended troubleshooting step as already tried and recommend ESCALATE. Also escalate if account access is needed, a wider outage may exist, or safety/billing is involved. Use the earlier chain outputs and do not invent facts.\nCustomer: " + message + "\nClassification: " + classification + "\nMissing information: " + missing + "\nSolution: " + solution
    escalation = ask_openrouter(escalation_prompt)
    return original_prompt, improved_prompt, classification, missing, solution, escalation
original_prompt, improved_prompt, classification, missing, solution, escalation = run_chain(customer_message)
print("PROVIDER: OpenRouter\nMODEL:", OPENROUTER_MODEL)
print("\nCUSTOMER MESSAGE:\n", customer_message)
print("\nORIGINAL PROMPT:\n", original_prompt)
print("\nIMPROVED PROMPT:\n", improved_prompt)
print("\nSTEP 1 - CLASSIFICATION:\n", classification)
print("\nSTEP 2 - MISSING INFORMATION (max 3 questions requested):\n", missing)
print("\nSTEP 3 - PROPOSED SOLUTION:\n", solution)
print("\nSTEP 4 - ESCALATION DECISION:\n", escalation)
print("\nITERATION NOTE: The improved prompts add explicit anti-hallucination rules, preserve the step-to-step inputs, cap follow-up questions at three, and explicitly escalate when the already-tried router restart meets the escalation rule.")
assert all([classification, missing, solution, escalation])
assert "ESCALATE" in escalation.upper()
print("\nSUCCESS: Four connected OpenRouter calls completed a tested prompt chain, including escalation after an already-tried troubleshooting step.")

PROVIDER: OpenRouter
MODEL: openai/gpt-4o-mini

CUSTOMER MESSAGE:
 My internet has not worked since yesterday, and I already restarted my router.

ORIGINAL PROMPT:
 Classify the support issue by category, urgency, and emotional tone. Do not invent account details, refunds, or policies. Return concise labeled fields.
Customer: My internet has not worked since yesterday, and I already restarted my router.

IMPROVED PROMPT:
 Classify the support issue by category, urgency, and emotional tone. Use only the message; do not invent account details, refunds, policies, outage status, or troubleshooting results. Return exactly CATEGORY, URGENCY, and TONE with one short reason each.
Customer: My internet has not worked since yesterday, and I already restarted my router.

STEP 1 - CLASSIFICATION:
 CATEGORY: Technical Issue  
URGENCY: High  
TONE: Frustrated  

Reason: The customer is experiencing a service disruption and has already attempted a common troubleshooting step, indicating a need for pr

In [13]:
# Provider-only smoke test for Exercise 1
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
response = client.chat.completions.create(model="openai/gpt-4o-mini", messages=[{"role":"user","content":"Reply with exactly: OPENROUTER_OK"}], temperature=0)
print("OPENROUTER SMOKE TEST:", response.choices[0].message.content.strip())

OPENROUTER SMOKE TEST: OPENROUTER_OK
